In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import re
import seaborn as sns
from math import radians, sin, cos, asin, sqrt
import imageio
import os
from IPython.display import Image, display
import time
from scipy.stats import gaussian_kde
from statistics import mean
import statistics
import random

In [2]:
plotdir = 'plots/'

In [3]:
#Loading all the loci to get the mean later
pattern = 'data/contemporary_nw_samples_only_extracted_*locus_10M_NoneT_2_3_4s_Nonet.forgotten_locs'

fns = sorted(glob.iglob(pattern))
starts = [int(re.search(r'extracted_(\d+)locus', fn).group(1)) for fn in fns]
locus_order = np.argsort(starts)
fnss_0 = [fns[i] for i in locus_order]
fnss_0.pop(0) # First locus is older version dont want it

# anc_locs_0 = []
# for i,fns in enumerate(fnss_0):
#     anc_loc = np.loadtxt(fns, delimiter=',')
#     anc_locs_0.append(anc_loc)

# anc_locs_0 = np.array(anc_locs_0)
# anc_locs_0 = np.swapaxes(anc_locs_0,0,1)
# L, n, T, d = anc_locs_0.shape
# L, n, T, d 

anc_locs_0 = []
for i, fpath in enumerate(fnss_0):
    df = pd.read_csv(fpath, header=None, names=["sample", "time", "lat", "lon"])
    samples = sorted(df["sample"].unique())
    timepoints = sorted(df["time"].unique())

    n = len(samples)
    T = len(timepoints)

    if n * T != len(df):
        print(f"Skipping {fpath}: expected {n*T} rows but got {len(df)}")
        continue
    sample_map = {v: i for i, v in enumerate(samples)}
    time_map = {v: i for i, v in enumerate(timepoints)}

    # Map samples and times to indices
    s_idx = df["sample"].map(sample_map).values
    t_idx = df["time"].map(time_map).values

    # Pre-allocate array
    arr = np.full((n, T, 2), np.nan)

    # Fill lat/lon efficiently using broadcasting
    arr[s_idx, t_idx, 0] = df["lon"].values
    arr[s_idx, t_idx, 1] = df["lat"].values

    anc_locs_0.append(arr)

anc_locs_0 = np.array(anc_locs_0)  # shape: (L, n, T, 2)
print("Final shape:", anc_locs_0.shape)

anc_locs = anc_locs_0

L, n, T, d = anc_locs.shape
L, n, T, d

Final shape: (995, 3, 1, 2)


(995, 3, 1, 2)